# F1AeroNet — Evaluation & Visualisation

**Standalone evaluation notebook** for F1AeroNet, a Gauge Equivariant Mesh CNN (GEM-CNN)  
that predicts aerodynamic fields (Cp, WSS, Cd) on 3D car surfaces from geometry alone.

## What this notebook does

1. Loads `runs_200/best.pt` (model checkpoint — no training required).
2. Loads the DrivAerNet++ test split from `data/drivaernet_real` (falls back to synthetic data if missing).
3. Runs inference on every sample, recording wall-clock time.
4. Computes metrics: Cp MAE/RMSE/R², Cd MAE / RelErr%, inference speedup vs CFD.
5. Produces all assets consumed by `Demo_project/demo.html`:
   - `outputs/viz/loss_curves.png`
   - `outputs/viz/scatter_all.png`
   - `outputs/viz/error_histograms.png`
   - Per-sample `.vtp` files with `Cp`, `WallShearStress`, etc.
   - Per-sample 3-view PNG renders (Cp predicted, WSS magnitude, Cp true vs predicted).
   - `outputs/viz/eval_summary.json` — copy-paste values for `demo.html`.

**Run from the project root** (`f1_aero_gem/`) so that relative paths resolve correctly.

In [ ]:
# ── Cell 1 · Imports ─────────────────────────────────────────────────────────
import os
import sys
import json
import time
import warnings
import numpy as np
import pandas as pd

# Ensure project root is on the path so local modules resolve
sys.path.insert(0, os.path.abspath('.'))

import torch
import yaml

import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe in all environments
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
HAS_MPL = True

from models.f1_net import F1AeroNet
from data.drivaernet_dataset import DrivAerNetDataset, make_synthetic_dataset

try:
    import pyvista as pv
    pv.start_xvfb()            # headless rendering on servers; safe no-op on desktops
    HAS_PYVISTA = True
    print("[INFO] pyvista available — VTP export and off-screen renders enabled.")
except Exception:
    HAS_PYVISTA = False
    print("[WARNING] pyvista not found or Xvfb unavailable — VTP export disabled.")
    print("          pip install pyvista vtk  to enable.")

print(f"[INFO] torch {torch.__version__}  |  numpy {np.__version__}")

In [ ]:
# ── Cell 2 · Config / Paths / Physical Constants ──────────────────────────────

# ── Paths ────────────────────────────────────────────────────────────────────
CHECKPOINT  = 'new_final_run/best.pt'
CONFIG_YAML = 'configs/f1_base.yaml'    # used only as a fallback; cfg is read from checkpoint
DATA_ROOT   = 'data/drivaernet_real'    # ← update if your data lives elsewhere
OUT_DIR     = 'outputs/viz'
TRAIN_LOG   = 'new_final_run/training_log.csv'

os.makedirs(OUT_DIR, exist_ok=True)

# ── Physical constants (SI) ───────────────────────────────────────────────────
RHO     = 1.225            # kg/m³  — air density at sea level
U_INF   = 83.33            # m/s    — ~300 km/h free-stream velocity
MU      = 1.81e-5          # Pa·s   — dynamic viscosity of air at 20°C
L_REF   = 4.6              # m      — reference car length
TAU_REF = MU * U_INF / L_REF   # ~3.27e-4 Pa  — WSS reference scale
Q_INF   = 0.5 * RHO * U_INF**2 # ~4248 Pa     — free-stream dynamic pressure

CFD_HOURS = 14.0           # approximate CFD wall-clock hours for one car design

# ── Device: prefer MPS (Apple Silicon) → CUDA → CPU ─────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f"[INFO] Device : {DEVICE}")
print(f"[INFO] OUT_DIR: {os.path.abspath(OUT_DIR)}")

In [ ]:
# ── Cell 3 · Load Model from Checkpoint ───────────────────────────────────────

if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT}\n"
        "Please place runs_200/best.pt in the project root."
    )

print(f"[INFO] Loading checkpoint: {CHECKPOINT}")
ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)

# ── Read config from checkpoint (robust: avoids yaml/checkpoint mismatch) ────
if 'cfg' in ckpt:
    saved_cfg = ckpt['cfg']
    # Kaggle training saves the full config; extract the model subsection
    cfg = saved_cfg.get('model', saved_cfg)
    print(f"[INFO] Config loaded from checkpoint (epoch {ckpt.get('epoch', '?')}, "
          f"best_val={ckpt.get('best_val', float('nan')):.6f})")
elif os.path.exists(CONFIG_YAML):
    with open(CONFIG_YAML) as fh:
        full_yaml = yaml.safe_load(fh)
    cfg = full_yaml.get('model', full_yaml)
    print(f"[WARNING] 'cfg' not in checkpoint — falling back to {CONFIG_YAML}")
else:
    cfg = {}   # F1AeroNet.from_config uses sensible defaults
    print("[WARNING] No config found — using F1AeroNet defaults.")

model = F1AeroNet.from_config(cfg)
model.load_state_dict(ckpt['model'])
model = model.to(DEVICE)
model.eval()

param_counts = model.count_parameters()
print(f"[INFO] Model loaded — {param_counts['total']:,} trainable parameters")
print(f"       GEM blocks: {param_counts['gem_blocks']:,}  |  "
      f"cp_head: {param_counts['cp_head']:,}  |  "
      f"cd_head: {param_counts['cd_head']:,}")

In [ ]:
# ── Cell 4 · Load Cd Normalisation Stats ──────────────────────────────────────

cd_stats_path = os.path.join(DATA_ROOT, 'cd_stats.json')

if os.path.exists(cd_stats_path):
    with open(cd_stats_path) as fh:
        cd_stats = json.load(fh)
    CD_MEAN = float(cd_stats['cd_mean'])
    CD_STD  = float(cd_stats['cd_std'])
    print(f"[INFO] Cd stats loaded: mean={CD_MEAN:.4f}  std={CD_STD:.4f}")
else:
    # Compute stats on-the-fly from the training split cached files, if available
    cache_train_dir = os.path.join(DATA_ROOT, 'processed', 'train')
    cds = []
    if os.path.isdir(cache_train_dir):
        for fn in os.listdir(cache_train_dir):
            if fn.endswith('.pt'):
                try:
                    d = torch.load(os.path.join(cache_train_dir, fn), weights_only=False)
                    if hasattr(d, 'y_cd') and d.y_cd is not None:
                        cds.append(float(d.y_cd.reshape(-1)[0]))
                except Exception:
                    pass

    if len(cds) >= 2:
        t = torch.tensor(cds, dtype=torch.float32)
        CD_MEAN = float(t.mean())
        CD_STD  = float(t.std())
        print(f"[INFO] Cd stats computed from {len(cds)} train samples: "
              f"mean={CD_MEAN:.4f}  std={CD_STD:.4f}")
        # Cache for next run
        with open(cd_stats_path, 'w') as fh:
            json.dump({'cd_mean': CD_MEAN, 'cd_std': CD_STD}, fh)
        print(f"[INFO] Saved to {cd_stats_path}")
    else:
        CD_MEAN, CD_STD = 0.0, 1.0
        print("[WARNING] cd_stats.json not found and no training cache available.")
        print("          Using CD_MEAN=0, CD_STD=1 — Cd predictions will be in normalised space.")

In [ ]:
# ── Cell 5 · Load Dataset ─────────────────────────────────────────────────────

USE_SYNTHETIC = False
dataset = None

if os.path.isdir(DATA_ROOT):
    # Try test split first, fall back to val
    for split_name in ('test', 'val'):
        try:
            ds = DrivAerNetDataset(DATA_ROOT, split=split_name)
            ds.set_cd_stats(CD_MEAN, CD_STD)
            if len(ds) > 0:
                dataset = ds
                SPLIT_NAME = split_name
                print(f"[INFO] Loaded '{split_name}' split: {len(ds)} samples")
                break
        except Exception as exc:
            print(f"[INFO] Could not load '{split_name}' split: {exc}")

if dataset is None or len(dataset) == 0:
    print("[WARNING] Real dataset unavailable — falling back to synthetic data.")
    dataset = make_synthetic_dataset(n_meshes=12, n_vertices=500)
    SPLIT_NAME = 'synthetic'
    USE_SYNTHETIC = True
    print(f"[INFO] Synthetic dataset: {len(dataset)} samples")

N_SAMPLES = len(dataset)
print(f"[INFO] Total samples to evaluate: {N_SAMPLES}")

In [ ]:
# ── Cell 6 · Denormalisation Utilities ───────────────────────────────────────

def symlog_inv(x: np.ndarray) -> np.ndarray:
    """Inverse symmetric-log: sign(x) * (exp(|x|) - 1)."""
    return np.sign(x) * (np.exp(np.abs(x)) - 1.0)


def denorm_cp(cp_norm: np.ndarray, sample) -> np.ndarray:
    """
    Denormalise predicted Cp from normalised space to physical dimensionless Cp.

    Pipeline (reversed):
      1. Unstandardise: cp_symlog = cp_norm * cp_sl_std + cp_sl_mean
      2. Inverse symlog: cp_phys = sign(cp_symlog) * (exp(|cp_symlog|) - 1)
    """
    cp_sl_mean = float(sample.cp_sl_mean) if hasattr(sample, 'cp_sl_mean') else 0.0
    cp_sl_std  = float(sample.cp_sl_std)  if hasattr(sample, 'cp_sl_std')  else 1.0
    cp_symlog  = cp_norm * cp_sl_std + cp_sl_mean
    return symlog_inv(cp_symlog)


def denorm_wss(wss_norm: np.ndarray, sample) -> np.ndarray:
    """
    Denormalise predicted WSS from normalised space to physical Pa.

    Pipeline (reversed):
      1. Unstandardise per-component: wss_symlog = wss_norm * wss_sl_std + wss_sl_mean
         where wss_sl_mean/std are shape (3,)
      2. Inverse symlog → wss in units of tau_ref
      3. Multiply by TAU_REF to get Pa
    """
    if hasattr(sample, 'wss_sl_mean') and sample.wss_sl_mean is not None:
        wss_mean = sample.wss_sl_mean.numpy().reshape(1, 3)   # (1, 3)
        wss_std  = sample.wss_sl_std.numpy().reshape(1, 3)    # (1, 3)
    else:
        wss_mean = np.zeros((1, 3), dtype=np.float32)
        wss_std  = np.ones((1, 3),  dtype=np.float32)

    wss_symlog = wss_norm * wss_std + wss_mean   # (V, 3) in symlog(tau_ref) space
    wss_nd     = symlog_inv(wss_symlog)           # (V, 3) dimensionless (scaled by tau_ref)
    return wss_nd * TAU_REF                       # (V, 3) in Pa


def denorm_cd(cd_norm_val: float) -> float:
    """Denormalise predicted Cd from z-score space to physical drag coefficient."""
    return cd_norm_val * CD_STD + CD_MEAN


print("[INFO] Denormalisation utilities defined.")
print(f"       TAU_REF = {TAU_REF:.4e} Pa   |   Q_INF = {Q_INF:.2f} Pa")

In [ ]:
# ── Cell 7 · Inference Loop ────────────────────────────────────────────────────

def _sync():
    """Synchronise GPU/MPS before timing to avoid measuring scheduling overhead."""
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elif DEVICE.type == 'mps':
        torch.mps.synchronize()


# ── Warm-up pass (avoids counting JIT/compile latency in timing) ─────────────
print("[INFO] Running warm-up pass ...")
with torch.no_grad():
    _sample = dataset[0] if not USE_SYNTHETIC else dataset[0]
    _x       = _sample.x.to(DEVICE)
    _ei      = _sample.edge_index.to(DEVICE)
    _ang     = _sample.edge_angles.to(DEVICE)
    _tr      = _sample.edge_transporters.to(DEVICE)
    _batch   = torch.zeros(_sample.num_nodes, dtype=torch.long, device=DEVICE)
    _ = model(x=_x, edge_index=_ei, angles=_ang, transporters=_tr, batch=_batch)
    _sync()
print("[INFO] Warm-up complete. Starting timed inference loop ...")

# ── Storage for results ───────────────────────────────────────────────────────
results = []   # list of dicts, one per sample

for idx in range(N_SAMPLES):
    sample = dataset[idx] if USE_SYNTHETIC else dataset[idx]
    did    = getattr(sample, 'design_id', f'sample_{idx:04d}')

    x            = sample.x.to(DEVICE)
    edge_index   = sample.edge_index.to(DEVICE)
    angles       = sample.edge_angles.to(DEVICE)
    transporters = sample.edge_transporters.to(DEVICE)
    batch        = torch.zeros(sample.num_nodes, dtype=torch.long, device=DEVICE)

    _sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        pred = model(x=x, edge_index=edge_index,
                     angles=angles, transporters=transporters,
                     batch=batch)
    _sync()
    t1 = time.perf_counter()
    elapsed_s = t1 - t0

    # ── Move predictions to CPU numpy ────────────────────────────────────────
    cp_norm_pred  = pred['cp'].cpu().numpy()           # (V,)   normalised
    wss_norm_pred = pred['wss'].cpu().numpy()          # (V, 3) normalised
    cd_norm_pred  = float(pred['cd'].cpu().item())

    # ── Ground truth (normalised) ─────────────────────────────────────────────
    cp_norm_true  = sample.y_cp.numpy() if sample.y_cp is not None else np.zeros_like(cp_norm_pred)
    wss_norm_true = sample.y_wss.numpy() if sample.y_wss is not None else np.zeros_like(wss_norm_pred)

    # y_cd may be normalised (if set_cd_stats was called) or raw
    cd_norm_true_val  = float(sample.y_cd.reshape(-1)[0])

    # ── Denormalise ────────────────────────────────────────────────────────────
    cp_phys_pred  = denorm_cp(cp_norm_pred,  sample)   # (V,)   dimensionless
    wss_phys_pred = denorm_wss(wss_norm_pred, sample)  # (V, 3) Pa
    cd_phys_pred  = denorm_cd(cd_norm_pred)

    cp_phys_true  = denorm_cp(cp_norm_true,  sample)   # (V,)   dimensionless
    wss_phys_true = denorm_wss(wss_norm_true, sample)  # (V, 3) Pa
    cd_phys_true  = denorm_cd(cd_norm_true_val)

    # ── Vertex geometry ───────────────────────────────────────────────────────
    vertices = sample.pos.numpy() if hasattr(sample, 'pos') else np.zeros((sample.num_nodes, 3))
    # face stored as (3, F) — transpose to (F, 3)
    if hasattr(sample, 'face') and sample.face is not None:
        faces = sample.face.numpy().T.astype(np.int64)   # (F, 3)
    else:
        faces = np.zeros((0, 3), dtype=np.int64)

    results.append({
        'design_id':        did,
        'idx':              idx,
        'elapsed_s':        elapsed_s,
        # normalised (for metrics)
        'cp_norm_pred':     cp_norm_pred,
        'cp_norm_true':     cp_norm_true,
        'wss_norm_pred':    wss_norm_pred,
        'wss_norm_true':    wss_norm_true,
        'cd_norm_pred':     cd_norm_pred,
        'cd_norm_true':     cd_norm_true_val,
        # physical (for viz)
        'cp_phys_pred':     cp_phys_pred,
        'cp_phys_true':     cp_phys_true,
        'wss_phys_pred':    wss_phys_pred,
        'wss_phys_true':    wss_phys_true,
        'cd_phys_pred':     cd_phys_pred,
        'cd_phys_true':     cd_phys_true,
        # geometry
        'vertices':         vertices,
        'faces':            faces,
        'n_vertices':       vertices.shape[0],
        'n_faces':          faces.shape[0],
    })

    print(f"  [{idx+1:3d}/{N_SAMPLES}]  {did:<28s}  "
          f"Cd_pred={cd_phys_pred:.4f}  Cd_true={cd_phys_true:.4f}  "
          f"t={elapsed_s*1000:.1f}ms  V={vertices.shape[0]}")

print(f"\n[INFO] Inference complete. {N_SAMPLES} samples processed.")

In [ ]:
# ── Cell 8 · Metrics ──────────────────────────────────────────────────────────

# ── Aggregate arrays ─────────────────────────────────────────────────────────
all_cp_pred   = np.concatenate([r['cp_norm_pred'] for r in results])
all_cp_true   = np.concatenate([r['cp_norm_true'] for r in results])
all_cd_pred   = np.array([r['cd_phys_pred'] for r in results])
all_cd_true   = np.array([r['cd_phys_true'] for r in results])
all_times_s   = np.array([r['elapsed_s']    for r in results])

# ── Cp metrics (normalised space — fair comparison across samples) ────────────
cp_err        = all_cp_pred - all_cp_true
cp_mae        = float(np.mean(np.abs(cp_err)))
cp_rmse       = float(np.sqrt(np.mean(cp_err**2)))
cp_ss_res     = float(np.sum(cp_err**2))
cp_ss_tot     = float(np.sum((all_cp_true - all_cp_true.mean())**2))
cp_r2         = 1.0 - cp_ss_res / (cp_ss_tot + 1e-12)

# ── Cd metrics (physical space) ───────────────────────────────────────────────
cd_err        = all_cd_pred - all_cd_true
cd_mae        = float(np.mean(np.abs(cd_err)))
cd_rel_pct    = float(np.mean(np.abs(cd_err) / (np.abs(all_cd_true) + 1e-8)) * 100.0)

# ── Timing & speedup ──────────────────────────────────────────────────────────
mean_inf_s    = float(np.mean(all_times_s))
median_inf_s  = float(np.median(all_times_s))
speedup       = CFD_HOURS * 3600.0 / (mean_inf_s + 1e-12)

print("=" * 60)
print("  F1AeroNet Evaluation Summary")
print("=" * 60)
print(f"  Split              : {SPLIT_NAME}")
print(f"  Samples            : {N_SAMPLES}")
print(f"  Total vertices     : {sum(r['n_vertices'] for r in results):,}")
print()
print("  Cp (normalised space)")
print(f"    MAE              : {cp_mae:.4f}")
print(f"    RMSE             : {cp_rmse:.4f}")
print(f"    R²               : {cp_r2:.4f}")
print()
print("  Cd (physical space)")
print(f"    MAE              : {cd_mae:.4f}")
print(f"    Mean RelErr      : {cd_rel_pct:.2f}%")
print()
print("  Inference timing")
print(f"    Mean             : {mean_inf_s*1000:.1f} ms")
print(f"    Median           : {median_inf_s*1000:.1f} ms")
print(f"    vs CFD ({CFD_HOURS}h)   : {speedup:,.0f}× faster")
print("=" * 60)

METRICS = {
    'cp_mae':       cp_mae,
    'cp_rmse':      cp_rmse,
    'cp_r2':        cp_r2,
    'cd_mae':       cd_mae,
    'cd_rel_pct':   cd_rel_pct,
    'mean_inf_ms':  mean_inf_s * 1000.0,
    'median_inf_ms': median_inf_s * 1000.0,
    'speedup':      speedup,
    'n_samples':    N_SAMPLES,
    'split':        SPLIT_NAME,
}

In [ ]:
# ── Cell 9 · Training Loss Curve → loss_curves.png ───────────────────────────

if not HAS_MPL:
    print("[WARNING] matplotlib unavailable — skipping loss_curves.png")
else:
    LOG_PATH = TRAIN_LOG
    if not os.path.exists(LOG_PATH):
        print(f"[WARNING] Training log not found: {LOG_PATH} — skipping loss_curves.png")
    else:
        log_df = pd.read_csv(LOG_PATH)

        # Columns: epoch, lr, train_total, train_cp, train_wss, train_cd, train_cl,
        #          val_total, val_cp, val_wss, val_cd, val_cl, time_s
        epochs     = log_df['epoch'].values
        train_loss = log_df['train_total'].values
        val_loss   = log_df['val_total'].values

        best_epoch_idx = int(np.argmin(val_loss))
        best_epoch     = int(epochs[best_epoch_idx])
        best_val       = float(val_loss[best_epoch_idx])

        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor='#111111')

        for ax in axes:
            ax.set_facecolor('#1a1a2e')
            for spine in ax.spines.values():
                spine.set_color('#444')
            ax.tick_params(colors='#ccc')
            ax.yaxis.label.set_color('#ccc')
            ax.xaxis.label.set_color('#ccc')
            ax.title.set_color('#eee')

        # ── Left panel: total loss ─────────────────────────────────────────────
        ax = axes[0]
        ax.plot(epochs, train_loss, color='#4fc3f7', linewidth=1.5, label='Train')
        ax.plot(epochs, val_loss,   color='#ef9a9a', linewidth=1.5, label='Val')
        ax.scatter([best_epoch], [best_val], color='gold', s=120, zorder=5,
                   marker='*', label=f'Best (ep {best_epoch}, val={best_val:.4f})')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Total Loss')
        ax.set_title('Training Loss Curve')
        ax.legend(framealpha=0.3, labelcolor='white', facecolor='#222')
        ax.grid(True, alpha=0.2, color='#555')

        # ── Right panel: per-component val loss ───────────────────────────────
        ax = axes[1]
        component_colors = {'val_cp': '#80cbc4', 'val_wss': '#ce93d8', 'val_cd': '#ffcc80'}
        component_labels = {'val_cp': 'Val Cp', 'val_wss': 'Val WSS', 'val_cd': 'Val Cd'}
        for col, color in component_colors.items():
            if col in log_df.columns:
                ax.plot(epochs, log_df[col].values, color=color,
                        linewidth=1.4, label=component_labels[col])
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Component Loss')
        ax.set_title('Val Loss by Component')
        ax.legend(framealpha=0.3, labelcolor='white', facecolor='#222')
        ax.grid(True, alpha=0.2, color='#555')

        fig.suptitle('F1AeroNet — Training Diagnostics',
                     color='white', fontsize=13, y=1.02)
        plt.tight_layout()
        out_path = os.path.join(OUT_DIR, 'loss_curves.png')
        fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#111111')
        plt.close(fig)
        print(f"[INFO] Saved: {out_path}")

        # Store arrays for eval_summary.json
        TRAIN_LOSS_EPOCHS = epochs.tolist()
        TRAIN_LOSS_TRAIN  = [round(float(v), 6) for v in train_loss]
        TRAIN_LOSS_VAL    = [round(float(v), 6) for v in val_loss]
        BEST_EPOCH        = best_epoch
        BEST_VAL_LOSS     = best_val
        print(f"[INFO] Best epoch: {best_epoch}  val_total={best_val:.6f}")

In [ ]:
# ── Cell 10 · Cd Scatter Plot → scatter_all.png ────────────────────────────────

if not HAS_MPL:
    print("[WARNING] matplotlib unavailable — skipping scatter_all.png")
else:
    fig, ax = plt.subplots(figsize=(6, 6), facecolor='#111111')
    ax.set_facecolor('#1a1a2e')
    for spine in ax.spines.values():
        spine.set_color('#444')
    ax.tick_params(colors='#ccc')

    ax.scatter(all_cd_true, all_cd_pred,
               c='#4fc3f7', alpha=0.75, s=40, edgecolors='white',
               linewidths=0.3, zorder=3, label='Predictions')

    # y = x diagonal
    _min = float(min(all_cd_true.min(), all_cd_pred.min())) * 0.97
    _max = float(max(all_cd_true.max(), all_cd_pred.max())) * 1.03
    ax.plot([_min, _max], [_min, _max], '--', color='gold',
            linewidth=1.2, label='y = x (perfect)', zorder=2)

    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(_min, _max)
    ax.set_ylim(_min, _max)
    ax.set_xlabel('CFD Cd (ground truth)', color='#ccc')
    ax.set_ylabel('GEM-CNN Cd (predicted)', color='#ccc')
    ax.set_title('Drag Coefficient: CFD vs GEM-CNN', color='#eee')

    # Annotate MAE
    ax.text(0.05, 0.93,
            f'MAE = {cd_mae:.4f}\nRelErr = {cd_rel_pct:.1f}%',
            transform=ax.transAxes, color='#ffcc80', fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#222', alpha=0.7))

    ax.legend(framealpha=0.3, labelcolor='white', facecolor='#222')
    ax.grid(True, alpha=0.2, color='#555')

    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, 'scatter_all.png')
    fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#111111')
    plt.close(fig)
    print(f"[INFO] Saved: {out_path}")

    # Store for eval_summary.json
    CD_SCATTER_TRUE = [round(float(v), 6) for v in all_cd_true]
    CD_SCATTER_PRED = [round(float(v), 6) for v in all_cd_pred]

In [ ]:
# ── Cell 11 · Error Histograms → error_histograms.png ─────────────────────────

if not HAS_MPL:
    print("[WARNING] matplotlib unavailable — skipping error_histograms.png")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), facecolor='#111111')

    for ax in axes:
        ax.set_facecolor('#1a1a2e')
        for spine in ax.spines.values():
            spine.set_color('#444')
        ax.tick_params(colors='#ccc')
        ax.yaxis.label.set_color('#ccc')
        ax.xaxis.label.set_color('#ccc')
        ax.title.set_color('#eee')

    # ── Left: Cp absolute error distribution (normalised space) ──────────────
    ax = axes[0]
    cp_abs_err = np.abs(all_cp_pred - all_cp_true)
    ax.hist(cp_abs_err, bins=60, color='#4fc3f7', alpha=0.8, edgecolor='none',
            density=True)
    ax.axvline(np.mean(cp_abs_err), color='gold', linewidth=1.5, linestyle='--',
               label=f'Mean = {np.mean(cp_abs_err):.3f}')
    ax.axvline(np.median(cp_abs_err), color='#ef9a9a', linewidth=1.5, linestyle=':',
               label=f'Median = {np.median(cp_abs_err):.3f}')
    ax.set_xlabel('|Cp error| (normalised space)')
    ax.set_ylabel('Density')
    ax.set_title('Cp Absolute Error Distribution')
    ax.legend(framealpha=0.3, labelcolor='white', facecolor='#222')
    ax.grid(True, alpha=0.2, color='#555')

    # ── Right: Cd absolute error distribution (physical space) ───────────────
    ax = axes[1]
    cd_abs_err = np.abs(all_cd_pred - all_cd_true)
    n_bins = max(10, N_SAMPLES // 2)
    ax.hist(cd_abs_err, bins=n_bins, color='#ce93d8', alpha=0.8, edgecolor='none',
            density=True)
    ax.axvline(np.mean(cd_abs_err), color='gold', linewidth=1.5, linestyle='--',
               label=f'Mean = {np.mean(cd_abs_err):.4f}')
    ax.axvline(np.median(cd_abs_err), color='#ef9a9a', linewidth=1.5, linestyle=':',
               label=f'Median = {np.median(cd_abs_err):.4f}')
    ax.set_xlabel('|Cd error| (physical, dimensionless)')
    ax.set_ylabel('Density')
    ax.set_title('Cd Absolute Error Distribution')
    ax.legend(framealpha=0.3, labelcolor='white', facecolor='#222')
    ax.grid(True, alpha=0.2, color='#555')

    fig.suptitle('F1AeroNet — Prediction Error Histograms',
                 color='white', fontsize=13, y=1.02)
    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, 'error_histograms.png')
    fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#111111')
    plt.close(fig)
    print(f"[INFO] Saved: {out_path}")

In [ ]:
# ── Cell 12 · Helper Functions for Per-Sample Visualisation ──────────────────

def export_vtp(r: dict, out_dir: str) -> str:
    """
    Export prediction results for one sample to a VTP file.

    Fields written (names match demo.html JS expectations):
      Cp                — predicted physical Cp (V,)
      WallShearStress   — predicted physical WSS vector in Pa (V, 3)
      Cp_true           — ground-truth physical Cp (V,)
      Cp_error          — absolute Cp error in normalised space (V,)
      WSS_mag_pred      — predicted WSS magnitude (V,)
      WSS_mag_true      — ground-truth WSS magnitude (V,)

    FieldData:
      Cd_total          — predicted physical Cd (scalar)

    Returns the output file path.
    """
    if not HAS_PYVISTA:
        return ''

    vertices = r['vertices'].astype(np.float64)
    faces    = r['faces']
    n_faces  = faces.shape[0]

    if n_faces == 0:
        print(f"  [SKIP VTP] {r['design_id']}: no faces")
        return ''

    # Build pyvista-compatible face array: [3, i0, i1, i2, 3, i0, ...]
    pv_faces = np.hstack([
        np.full((n_faces, 1), 3, dtype=np.int64),
        faces.astype(np.int64)
    ]).flatten()

    mesh = pv.PolyData(vertices, pv_faces)

    # ── Point data ────────────────────────────────────────────────────────────
    cp_pred = r['cp_phys_pred'].astype(np.float32)
    cp_true = r['cp_phys_true'].astype(np.float32)
    cp_err  = np.abs(r['cp_norm_pred'] - r['cp_norm_true']).astype(np.float32)

    wss_pred = r['wss_phys_pred'].astype(np.float32)   # (V, 3)
    wss_true = r['wss_phys_true'].astype(np.float32)   # (V, 3)

    wss_mag_pred = np.linalg.norm(wss_pred, axis=1).astype(np.float32)
    wss_mag_true = np.linalg.norm(wss_true, axis=1).astype(np.float32)

    # Demo-HTML-facing names (must match exactly)
    mesh.point_data['Cp']               = cp_pred
    mesh.point_data['WallShearStress']  = wss_pred
    # Comparison arrays
    mesh.point_data['Cp_true']          = cp_true
    mesh.point_data['Cp_error']         = cp_err
    mesh.point_data['WSS_mag_pred']     = wss_mag_pred
    mesh.point_data['WSS_mag_true']     = wss_mag_true

    # ── Field data (scalars) ──────────────────────────────────────────────────
    mesh.field_data['Cd_total'] = np.array([r['cd_phys_pred']], dtype=np.float32)

    out_path = os.path.join(out_dir, f"{r['design_id']}_predictions.vtp")
    mesh.save(out_path)
    return out_path


def _make_tri_or_scatter(ax, verts, vals, faces, cmap, vmin, vmax, title,
                          xdim=0, ydim=1, xlabel='', ylabel=''):
    """
    Render a field on a projected 2D view using Triangulation.
    Falls back to scatter if triangulation fails (degenerate mesh).
    """
    x_proj = verts[:, xdim]
    y_proj = verts[:, ydim]

    drawn = False
    if faces.shape[0] > 0:
        try:
            triang = mtri.Triangulation(x_proj, y_proj, faces)
            tc = ax.tripcolor(triang, vals, cmap=cmap, vmin=vmin, vmax=vmax,
                              shading='gouraud', rasterized=True)
            drawn = True
        except Exception:
            drawn = False

    if not drawn:
        sc = ax.scatter(x_proj, y_proj, c=vals, cmap=cmap,
                        vmin=vmin, vmax=vmax, s=1.0, rasterized=True)
        tc = sc

    ax.set_title(title, fontsize=9, color='#ccc')
    ax.set_xlabel(xlabel, fontsize=7, color='#aaa')
    ax.set_ylabel(ylabel, fontsize=7, color='#aaa')
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='#aaa', labelsize=6)
    for spine in ax.spines.values():
        spine.set_color('#444')
    return tc


def _three_view_fig(r, vals, cmap, fig_title, vmin=None, vmax=None):
    """
    Create a 3-panel figure: side (x-z), top (x-y), front (y-z) views.
    Returns the figure.
    """
    verts = r['vertices']   # (V, 3) in world coords
    faces = r['faces']      # (F, 3)

    if vmin is None:
        vmin = float(np.percentile(vals, 2))
    if vmax is None:
        vmax = float(np.percentile(vals, 98))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='#111111')

    panels = [
        (0, 2, 'Side (x-z)',  'x', 'z'),
        (0, 1, 'Top (x-y)',   'x', 'y'),
        (1, 2, 'Front (y-z)', 'y', 'z'),
    ]

    tc = None
    for ax, (xd, yd, title, xl, yl) in zip(axes, panels):
        tc = _make_tri_or_scatter(ax, verts, vals, faces, cmap, vmin, vmax,
                                   title, xdim=xd, ydim=yd,
                                   xlabel=xl, ylabel=yl)

    if tc is not None:
        cbar = fig.colorbar(tc, ax=axes, fraction=0.02, pad=0.02)
        cbar.ax.yaxis.set_tick_params(color='#ccc')
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#ccc', fontsize=7)

    did = r['design_id']
    fig.suptitle(f"{fig_title} — {did}", color='white', fontsize=11, y=1.01)
    plt.tight_layout()
    return fig


def plot_cp_3view(r: dict, out_dir: str) -> str:
    """3-panel Cp predicted — side, top, front.  Returns output path."""
    if not HAS_MPL:
        return ''
    vals = r['cp_phys_pred']
    fig  = _three_view_fig(r, vals, cmap='RdBu_r',
                            fig_title='Cp Predicted (physical)')
    out_path = os.path.join(out_dir, f"{r['design_id']}_cp_predicted_3view.png")
    fig.savefig(out_path, dpi=130, bbox_inches='tight', facecolor='#111111')
    plt.close(fig)
    return out_path


def plot_wss_3view(r: dict, out_dir: str) -> str:
    """3-panel WSS magnitude — side, top, front.  Returns output path."""
    if not HAS_MPL:
        return ''
    vals = np.linalg.norm(r['wss_phys_pred'], axis=1)
    fig  = _three_view_fig(r, vals, cmap='hot',
                            fig_title='WSS Magnitude Predicted (Pa)')
    out_path = os.path.join(out_dir, f"{r['design_id']}_wss.png")
    fig.savefig(out_path, dpi=130, bbox_inches='tight', facecolor='#111111')
    plt.close(fig)
    return out_path


def plot_cp_comparison(r: dict, out_dir: str) -> str:
    """
    Side-by-side: Cp true vs Cp predicted, side view only.
    Returns output path.
    """
    if not HAS_MPL:
        return ''

    verts = r['vertices']
    faces = r['faces']
    cp_true = r['cp_phys_true']
    cp_pred = r['cp_phys_pred']

    # Shared colour range
    combined = np.concatenate([cp_true, cp_pred])
    vmin = float(np.percentile(combined, 2))
    vmax = float(np.percentile(combined, 98))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor='#111111')

    tc = None
    for ax, vals, title in zip(axes,
                                [cp_true, cp_pred],
                                ['Cp True (CFD)', 'Cp Predicted (GEM-CNN)']):
        tc = _make_tri_or_scatter(ax, verts, vals, faces,
                                   cmap='RdBu_r', vmin=vmin, vmax=vmax,
                                   title=title, xdim=0, ydim=2,
                                   xlabel='x', ylabel='z')

    if tc is not None:
        cbar = fig.colorbar(tc, ax=axes, fraction=0.02, pad=0.02)
        cbar.ax.yaxis.set_tick_params(color='#ccc')
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#ccc', fontsize=7)
        cbar.set_label('Cp (dimensionless)', color='#ccc', fontsize=8)

    did = r['design_id']
    fig.suptitle(f"Cp Comparison — {did}  |  Cd_pred={r['cd_phys_pred']:.4f}",
                 color='white', fontsize=10, y=1.01)
    plt.tight_layout()
    out_path = os.path.join(out_dir, f"{r['design_id']}_cp.png")
    fig.savefig(out_path, dpi=130, bbox_inches='tight', facecolor='#111111')
    plt.close(fig)
    return out_path


print("[INFO] Helper functions defined: export_vtp, plot_cp_3view, plot_wss_3view, plot_cp_comparison")

In [ ]:
# ── Cell 13 · Per-Sample Visualisation Loop ────────────────────────────────────

demo_metadata = []

for r in results:
    did = r['design_id']
    print(f"\n[VIZ] {did}  (V={r['n_vertices']}, F={r['n_faces']})")

    # ── VTP export ───────────────────────────────────────────────────────────
    vtp_path = ''
    if HAS_PYVISTA:
        try:
            vtp_path = export_vtp(r, OUT_DIR)
            if vtp_path:
                print(f"  VTP → {os.path.basename(vtp_path)}")
        except Exception as exc:
            print(f"  [WARN] VTP export failed: {exc}")

    # ── PNG exports ──────────────────────────────────────────────────────────
    cp3v_path = cmp_path = wss_path = ''

    if HAS_MPL:
        try:
            cp3v_path = plot_cp_3view(r, OUT_DIR)
            print(f"  Cp 3-view  → {os.path.basename(cp3v_path)}")
        except Exception as exc:
            print(f"  [WARN] Cp 3-view failed: {exc}")

        try:
            wss_path = plot_wss_3view(r, OUT_DIR)
            print(f"  WSS 3-view → {os.path.basename(wss_path)}")
        except Exception as exc:
            print(f"  [WARN] WSS 3-view failed: {exc}")

        try:
            cmp_path = plot_cp_comparison(r, OUT_DIR)
            print(f"  Cp compare → {os.path.basename(cmp_path)}")
        except Exception as exc:
            print(f"  [WARN] Cp comparison failed: {exc}")

    # ── Summary stats for this design ────────────────────────────────────────
    cp_pred = r['cp_norm_pred']
    cp_true = r['cp_norm_true']
    wss_mag = np.linalg.norm(r['wss_phys_pred'], axis=1)

    demo_metadata.append({
        'design_id':            did,
        'n_vertices':           int(r['n_vertices']),
        'n_faces':              int(r['n_faces']),
        'cd_pred':              round(float(r['cd_phys_pred']), 5),
        'cd_true':              round(float(r['cd_phys_true']), 5),
        'cd_err':               round(float(abs(r['cd_phys_pred'] - r['cd_phys_true'])), 5),
        'cp_mae_norm':          round(float(np.mean(np.abs(cp_pred - cp_true))), 5),
        'cp_rmse_norm':         round(float(np.sqrt(np.mean((cp_pred - cp_true)**2))), 5),
        'cp_phys_min':          round(float(r['cp_phys_pred'].min()), 4),
        'cp_phys_max':          round(float(r['cp_phys_pred'].max()), 4),
        'wss_mag_mean_Pa':      round(float(wss_mag.mean()), 5),
        'wss_mag_max_Pa':       round(float(wss_mag.max()), 5),
        'inference_ms':         round(float(r['elapsed_s'] * 1000), 2),
        'vtp_file':             os.path.basename(vtp_path) if vtp_path else '',
        'cp_3view_png':         os.path.basename(cp3v_path) if cp3v_path else '',
        'wss_png':              os.path.basename(wss_path)  if wss_path  else '',
        'cp_comparison_png':    os.path.basename(cmp_path)  if cmp_path  else '',
    })

print(f"\n[INFO] Per-sample visualisation complete. {len(demo_metadata)} designs processed.")

In [ ]:
# ── Cell 14 · Build and Save eval_summary.json ────────────────────────────────

# ── Training log arrays (may have been set in Cell 9) ────────────────────────
_has_log = ('TRAIN_LOSS_EPOCHS' in dir())

summary = {
    "meta": {
        "generated_by":  "F1_Eval_Visualisation.ipynb",
        "checkpoint":    CHECKPOINT,
        "split":         SPLIT_NAME,
        "n_samples":     N_SAMPLES,
        "device":        str(DEVICE),
        "torch_version": torch.__version__
    },
    "model": {
        "total_params":   param_counts['total'],
        "gem_blocks":     param_counts['gem_blocks'],
        "cp_head":        param_counts['cp_head'],
        "cd_head":        param_counts['cd_head']
    },
    "physical_constants": {
        "rho":       RHO,
        "U_inf":     U_INF,
        "mu":        MU,
        "L_ref":     L_REF,
        "tau_ref":   round(TAU_REF, 8),
        "q_inf":     round(Q_INF, 4),
        "cfd_hours": CFD_HOURS
    },
    "metrics": {
        "cp_mae":        round(METRICS['cp_mae'], 6),
        "cp_rmse":       round(METRICS['cp_rmse'], 6),
        "cp_r2":         round(METRICS['cp_r2'], 6),
        "cd_mae":        round(METRICS['cd_mae'], 6),
        "cd_rel_pct":    round(METRICS['cd_rel_pct'], 4),
        "mean_inf_ms":   round(METRICS['mean_inf_ms'], 2),
        "median_inf_ms": round(METRICS['median_inf_ms'], 2),
        "speedup_x":     round(METRICS['speedup'], 0)
    },
    "cd_scatter": {
        "cd_true": CD_SCATTER_TRUE if 'CD_SCATTER_TRUE' in dir() else [],
        "cd_pred": CD_SCATTER_PRED if 'CD_SCATTER_PRED' in dir() else []
    },
    "training_log": {
        "epochs":     TRAIN_LOSS_EPOCHS if _has_log else [],
        "train_loss": TRAIN_LOSS_TRAIN  if _has_log else [],
        "val_loss":   TRAIN_LOSS_VAL    if _has_log else [],
        "best_epoch": BEST_EPOCH        if _has_log else None,
        "best_val":   round(BEST_VAL_LOSS, 6) if _has_log else None
    },
    "designs": demo_metadata
}

summary_path = os.path.join(OUT_DIR, 'eval_summary.json')
with open(summary_path, 'w') as fh:
    json.dump(summary, fh, indent=2)

print(f"[INFO] Saved: {summary_path}")

# ── Print copy-paste block for demo.html ─────────────────────────────────────
print()
print("=" * 60)
print("  Copy-paste values for Demo_project/demo.html")
print("=" * 60)
print()
print("// ── Model metrics ───────────────────────────────────────")
print(f"const GEM_CP_MAE   = {METRICS['cp_mae']:.4f};")
print(f"const GEM_CP_R2    = {METRICS['cp_r2']:.4f};")
print(f"const GEM_CD_MAE   = {METRICS['cd_mae']:.4f};")
print(f"const GEM_CD_REL   = {METRICS['cd_rel_pct']:.2f};  // %")
print(f"const GEM_INF_MS   = {METRICS['mean_inf_ms']:.1f};  // ms per design")
print(f"const GEM_SPEEDUP  = {int(METRICS['speedup'])};  // x faster than CFD")
print()
print("// ── Car designs ──────────────────────────────────────────")
for m in demo_metadata:
    print(f"//   {m['design_id']}: Cd={m['cd_pred']:.4f} (true={m['cd_true']:.4f}), "
          f"Cp_MAE={m['cp_mae_norm']:.4f}, inf={m['inference_ms']:.1f}ms")
print()
print("// ── Output files ─────────────────────────────────────────")
all_files = []
for m in demo_metadata:
    for k in ('vtp_file', 'cp_3view_png', 'wss_png', 'cp_comparison_png'):
        if m[k]:
            all_files.append(m[k])
for fn in sorted(set(all_files)):
    print(f"//   outputs/viz/{fn}")
print(f"//   outputs/viz/loss_curves.png")
print(f"//   outputs/viz/scatter_all.png")
print(f"//   outputs/viz/error_histograms.png")
print(f"//   outputs/viz/eval_summary.json")
print("=" * 60)